# Census income classification with Keras

To download a copy of this notebook visit [github](https://github.com/shap/shap/tree/main/notebooks).

This notebook showcases the use of kernel and coalition explainer for explaining neural networks. 

In [ ]:
import matplotlib.pyplot as plt
from keras.layers import Dense, Dropout, Embedding, Flatten, Input, concatenate
from keras.models import Model
from sklearn.model_selection import train_test_split

import shap

# print the JS visualization code to the notebook
shap.initjs()

## Load dataset

In [ ]:
X, y = shap.datasets.adult()
X_display, y_display = shap.datasets.adult(display=True)

# normalize data (this is important for model convergence)
dtypes = list(zip(X.dtypes.index, map(str, X.dtypes)))
for k, dtype in dtypes:
    if dtype == "float32":
        X[k] -= X[k].mean()
        X[k] /= X[k].std()

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=7)

## Train Keras model

In [ ]:
# build model
input_els = []
encoded_els = []
for k, dtype in dtypes:
    input_els.append(Input(shape=(1,)))
    if dtype == "int8":
        e = Flatten()(Embedding(X_train[k].max() + 1, 1)(input_els[-1]))
    else:
        e = input_els[-1]
    encoded_els.append(e)
encoded_els = concatenate(encoded_els)
layer1 = Dropout(0.5)(Dense(100, activation="relu")(encoded_els))
out = Dense(1)(layer1)

# train model
regression = Model(inputs=input_els, outputs=[out])
regression.compile(optimizer="adam", loss="binary_crossentropy")
regression.fit(
    [X_train[k].values for k, t in dtypes],
    y_train,
    epochs=50,
    batch_size=512,
    shuffle=True,
    validation_data=([X_valid[k].values for k, t in dtypes], y_valid),
)

## Explain predictions

Here we take the Keras model trained above and explain why it makes different predictions for different individuals. SHAP expects model functions to take a 2D numpy array as input, so we define a wrapper function around the original Keras predict function.

In [ ]:
def f(X):
    return regression.predict([X[:, i] for i in range(X.shape[1])]).flatten()

### Explain a single prediction

Here we use a selection of 50 samples from the dataset to represent "typical" feature values, and then use 500 perturbation samples to estimate the SHAP values for a given prediction. Note that this requires 500 * 50 evaluations of the model.

In [ ]:
X.iloc[299, :]

In [ ]:
X.iloc[299:300, :]

In [ ]:
import numpy as np

f(np.array(X.iloc[299:300, :]))

In [ ]:
kernel_explainer = shap.KernelExplainer(f, X.iloc[:50, :])
shap_values = kernel_explainer.shap_values(X.iloc[299, :], nsamples=500)

shap.force_plot(kernel_explainer.expected_value, shap_values, X_display.iloc[299, :])

In [ ]:
shap_values = kernel_explainer(X.iloc[299:300, :])

Set up the Coalition Explainer.

In [ ]:
partition_tree = {
    "cluster_16": {
        "cluster_12": {
            "cluster_8": {"Age": "Age", "Workclass": "Workclass"},
            "cluster_9": {
                "Education-Num": "Education-Num",
                "Marital Status": "Marital Status",
            },
        },
        "cluster_13": {
            "cluster_10": {"Occupation": "Occupation", "Relationship": "Relationship"},
            "cluster_11": {"Race": "Race", "Sex": "Sex"},
        },
    },
    "cluster_17": {
        "cluster_14": {"Capital Gain": "Capital Gain", "Capital Loss": "Capital Loss"},
        "cluster_15": {"Hours per week": "Hours per week", "Country": "Country"},
    },
}

partition_tree = {
    "cluster_16": {
        "cluster_8": {"Age": "Age", "Race": "Race", "Sex": "Sex"},
    },
    "cluster_13": {
        "cluster_10": {
            "Occupation": "Occupation",
            "Hours per week": "Hours per week",
            "Relationship": "Relationship",
            "Education-Num": "Education-Num",
        },
        "cluster_14": {
            "Capital Gain": "Capital Gain",
            "Capital Loss": "Capital Loss",
            "Country": "Country",
        },
    },
}
partition_masker = shap.maskers.Partition(X.iloc[:50, :])

explainer = shap.explainers.Coalition(f, partition_masker, partition_tree=partition_tree)
row_to_explain = X.iloc[299:300, :]
winter_values = explainer(row_to_explain)

In [ ]:
f(np.array(X.iloc[299:300, :]))

In [ ]:
fig, ax = plt.subplots()
shap.plots.waterfall(shap_values[0], show=False)
plt.title("PartitionExplainer for instance 0 with the KernelExplainer")
plt.show()

In [ ]:
fig, ax = plt.subplots()
shap.plots.waterfall(winter_values[0], show=False)
plt.title("PartitionExplainer for instance 0 with the CoalitionExplainer")
plt.show()

In [ ]:
shap.initjs()
shap.force_plot(winter_values.base_values, winter_values.values, X_display.iloc[299:300, :])

### Explain many predictions

Here we repeat the above explanation process for 20 individuals. Since we are using a sampling based approximation each explanation can take a couple seconds depending on your machine setup.

In [ ]:
winter_values50 = explainer(X.iloc[320:330, :])

In [ ]:
shap_values50 = kernel_explainer.shap_values(X.iloc[320:330, :], nsamples=500)

In [ ]:
shap.force_plot(kernel_explainer.expected_value, shap_values50, X_display.iloc[320:330, :])

In [ ]:
shap.force_plot(winter_values50.base_values, winter_values50.values, X_display.iloc[320:330, :])